# 01. Exploratory Data Analysis & Financial Time Series Profiling
**Project**: Quantitative Deep Learning Architecture for S&P 500 Equities  
**Objective**: Ingest historical market data (2005–2025), inspect cross-sectional properties, evaluate return distributions, and examine cross-asset correlation structures under OSFI E-23 validation standards.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = "../data/sp500_universe.parquet"
MACRO_PATH = "../data/macro_indicators.parquet"

print("Loading ingested parquet datasets...")
stocks_df = pd.read_parquet(DATA_PATH)
macro_df = pd.read_parquet(MACRO_PATH)

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
macro_df['Date'] = pd.to_datetime(macro_df['Date'])

print(f"S&P 500 Master Records: {len(stocks_df):,}")
print(f"Unique Equities: {stocks_df['Ticker'].nunique()}")
print(f"Date Span: {stocks_df['Date'].min().date()} to {stocks_df['Date'].max().date()}")
display(stocks_df.head())

## 1. Asset Performance & Risk Profiling
Computing annualized log returns, annualized volatility, Sharpe ratios, and maximum drawdowns across all constituents.

In [ ]:
close_col = "Adj Close" if "Adj Close" in stocks_df.columns else "Close"
stats = []

for ticker, group in stocks_df.groupby('Ticker'):
    group = group.sort_values('Date').dropna(subset=[close_col])
    prices = group[close_col].values.astype(float)
    if len(prices) < 252:
        continue
    
    log_ret = np.log(prices[1:] / prices[:-1])
    log_ret = log_ret[np.isfinite(log_ret)]
    if len(log_ret) == 0:
        continue
        
    ann_return = float(np.mean(log_ret) * 252)
    ann_vol = float(np.std(log_ret, ddof=1) * np.sqrt(252))
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0
    
    cum_ret = np.cumprod(1 + log_ret)
    peak = np.maximum.accumulate(cum_ret)
    dd = (cum_ret - peak) / peak
    max_dd = float(np.min(dd)) if len(dd) > 0 else 0.0
    
    stats.append({
        "Ticker": ticker,
        "Observations": len(prices),
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    })

stats_df = pd.DataFrame(stats).sort_values("Sharpe Ratio", ascending=False).reset_index(drop=True)
print("Top 10 Equities Ranked by Historical Sharpe Ratio:")
display(stats_df.head(10))

## 2. Correlation Structure Analysis

In [ ]:
top_tickers = stats_df.head(10)['Ticker'].tolist()
pivot_df = stocks_df[stocks_df['Ticker'].isin(top_tickers)].pivot(index='Date', columns='Ticker', values=close_col)
returns_pivot = pivot_df.pct_change().dropna()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(returns_pivot.corr(), dtype=bool))
sns.heatmap(returns_pivot.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Daily Return Correlation Matrix (Top 10 Assets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()